# 03 — Trained MDN–LSTM probabilistic forecasting

## Goal
Train a **real PyTorch MDN–LSTM** on the first 60 days of the seven core AQUASURE water-quality variables and estimate an end-cycle harvest-health distribution.

This notebook deliberately removes the previous untrained NumPy fallback. PyTorch is required. The model is evaluated with **farm-grouped out-of-fold validation** so that cycles from the same farm do not leak across train/test partitions.

### Seven core inputs
Temperature, dissolved oxygen, pH, salinity, ammonia, nitrite, and turbidity.

### Forecast target
For each 120-day production cycle, the model sees Days 1–60 and models

\[
p(H\mid X_{1:60})=\sum_{k=1}^{K}\pi_k(X_{1:60})\,\mathcal N(H\mid\mu_k(X_{1:60}),\sigma_k^2(X_{1:60})),
\]

where \(H\) is end-cycle `harvest_health`. Day-60 PHRI is then

\[
\mathrm{PHRI}_{60}=P(H<H^*\mid X_{1:60}).
\]

### Scientific rationale
- LSTM is used as the recurrent encoder because its gated state was developed to improve learning over extended dependencies (Hochreiter & Schmidhuber, 1997).
- The mixture-density head models a conditional probability distribution rather than only a conditional mean (Bishop, 1994).
- Entire farms are separated across folds because repeated cycles from one farm are hierarchically related; structured cross-validation is preferable to random row-wise splitting when observations are grouped (Roberts et al., 2017).
- Validation loss is predictive negative log-likelihood, a proper density score (Gneiting & Raftery, 2007).
- Gradient-norm clipping is used to control exploding recurrent gradients (Pascanu et al., 2013).
- AdamW is used for optimization with decoupled weight decay (Loshchilov & Hutter, 2019).


In [1]:
from pathlib import Path
import json, math, random, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.special import ndtr
from scipy.stats import norm, kstest
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss, mean_squared_error, mean_absolute_error

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
ARTIFACTS = ROOT / "artifacts"
TRAINED = ROOT / "artifacts_trained"
TRAINED.mkdir(exist_ok=True)

SEED = 20260830
LOOKBACK = 60
CORE_VARIABLES = [
    "temperature_c", "dissolved_oxygen_mg_l", "ph", "salinity_ppt",
    "ammonia_mg_l", "nitrite_mg_l", "turbidity_ntu",
]
H_STAR = json.loads((ARTIFACTS / "biological_model_parameters.json").read_text())["H_star"]
print("PyTorch:", torch.__version__)
print("Core inputs:", CORE_VARIABLES)
print("H*:", H_STAR)


PyTorch: 2.10.0+cpu
Core inputs: ['temperature_c', 'dissolved_oxygen_mg_l', 'ph', 'salinity_ppt', 'ammonia_mg_l', 'nitrite_mg_l', 'turbidity_ntu']
H*: 0.7864821387045725


## 1. Assemble one 60-day sequence per production cycle

The grain is **one production cycle = one training example**. No temporal shuffling occurs inside the 60-day sequence.

In [2]:
pond = pd.read_csv(ARTIFACTS / "pond_timeseries.csv")
biology = pd.read_csv(ARTIFACTS / "biological_cycles.csv").set_index("cycle_id")

sequences, targets, labels, farm_ids, cycle_ids = [], [], [], [], []
for (farm_id, cycle_id), group in pond[pond.day <= LOOKBACK].groupby(["farm_id", "cycle_id"], sort=False):
    group = group.sort_values("day")
    if len(group) != LOOKBACK:
        continue
    sequences.append(group[CORE_VARIABLES].to_numpy(np.float32))
    targets.append(float(biology.loc[cycle_id, "harvest_health"]))
    labels.append(int(biology.loc[cycle_id, "severe_event"]))
    farm_ids.append(farm_id)
    cycle_ids.append(cycle_id)

X = np.stack(sequences)
y = np.asarray(targets, dtype=np.float32)
event = np.asarray(labels, dtype=int)
farm_ids = np.asarray(farm_ids)
cycle_ids = np.asarray(cycle_ids)
order = np.argsort(cycle_ids)
X, y, event, farm_ids, cycle_ids = X[order], y[order], event[order], farm_ids[order], cycle_ids[order]
print("X shape:", X.shape, "target shape:", y.shape)


X shape: (960, 60, 7) target shape: (960,)


## 2. MDN–LSTM architecture

The LSTM maps the \(60\times 7\) sequence to its final hidden representation. A small nonlinear head outputs:

\[
\pi_k\ge 0,\quad \sum_k\pi_k=1,\qquad 0<\mu_k<1,\qquad \sigma_k>0.
\]

The implementation uses `softmax` for mixture weights, `sigmoid` for component means because `harvest_health` is bounded in the prototype, and `softplus + 0.005` for positive, numerically stable scales.

In [3]:
def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

class MDNLSTM(nn.Module):
    def __init__(self, n_features, hidden_size=64, mixtures=2, dropout=0.10):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden_size, batch_first=True)
        self.norm = nn.LayerNorm(hidden_size)
        self.hidden = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Dropout(dropout),
        )
        self.pi = nn.Linear(hidden_size, mixtures)
        self.mu = nn.Linear(hidden_size, mixtures)
        self.sigma = nn.Linear(hidden_size, mixtures)

    def forward(self, sequence):
        encoded, _ = self.lstm(sequence)
        state = self.hidden(self.norm(encoded[:, -1, :]))
        log_pi = torch.log_softmax(self.pi(state), dim=-1)
        mu = torch.sigmoid(self.mu(state))
        sigma = torch.nn.functional.softplus(self.sigma(state)) + 0.005
        return log_pi, mu, sigma

def mdn_nll(target, log_pi, mu, sigma):
    target = target[:, None]
    log_normal = -0.5*math.log(2*math.pi) - torch.log(sigma) - 0.5*((target-mu)/sigma)**2
    return -torch.logsumexp(log_pi + log_normal, dim=1).mean()


In [4]:
# Set to True to reproduce the nested grouped training from scratch.
# This can take several minutes on CPU.
RERUN_TRAINING = False
if RERUN_TRAINING:
    %run ../scripts/train_mdn_lstm_nested.py
    %run ../scripts/train_final_mdn.py
else:
    print("Using the archived trained PyTorch artifacts in artifacts_trained/. Set RERUN_TRAINING=True to retrain.")

Using the archived trained PyTorch artifacts in artifacts_trained/. Set RERUN_TRAINING=True to retrain.


## 3. Training controls

The architecture is not chosen using the outer test farms. Within each outer fold, separate validation farms are used for early stopping and to compare five compact candidate architectures. This is a **nested grouped-validation design**.

Candidates:
- H16 / K2 / dropout 0.05
- H32 / K2 / dropout 0.10
- H64 / K2 / dropout 0.10
- H32 / K3 / dropout 0.10
- H32 / K1 / dropout 0.10

Common optimization: AdamW, learning rate \(10^{-3}\), weight decay \(10^{-4}\), gradient clipping at 1.0, `ReduceLROnPlateau`, maximum 150 epochs, patience 20.

In [5]:
CONFIGS = [
    {"name":"H16_K2_D05", "hidden":16, "K":2, "dropout":0.05},
    {"name":"H32_K2_D10", "hidden":32, "K":2, "dropout":0.10},
    {"name":"H64_K2_D10", "hidden":64, "K":2, "dropout":0.10},
    {"name":"H32_K3_D10", "hidden":32, "K":3, "dropout":0.10},
    {"name":"H32_K1_D10", "hidden":32, "K":1, "dropout":0.10},
]

# The full nested-search implementation and its saved evidence are included in
# scripts/train_mdn_lstm_nested.py. The archived run used six outer farm-grouped folds.
search = pd.read_csv(TRAINED / "mdn_lstm_nested_search.csv")
search.groupby("config").agg(mean_val_nll=("val_nll","mean"), median_val_nll=("val_nll","median"), mean_best_epoch=("best_epoch","mean")).sort_values("mean_val_nll")


,mean_val_nll,median_val_nll,mean_best_epoch
config,,,
H64_K2_D10,-2.049339,-2.041665,84.500000
H32_K1_D10,-2.019882,-2.043376,135.833333
H32_K3_D10,-1.956344,-1.995692,90.000000
H32_K2_D10,-1.955109,-1.987024,109.166667
H16_K2_D05,-1.919421,-1.915075,149.000000


## 4. Out-of-fold results from the actual PyTorch run

Every row below was predicted by a model that did **not** train on that row's farm. The output file is `artifacts_trained/mdn_lstm_oof_nested.csv`.

In [6]:
metrics = json.loads((TRAINED / "mdn_lstm_nested_metrics.json").read_text())
pd.Series(metrics)


n                                                                 960
input_features      [temperature_c, dissolved_oxygen_mg_l, ph, sal...
outer_folds                                                         6
rmse                                                         0.031897
mae                                                          0.025934
mean_crps                                                    0.018223
phri_auc                                                     0.947327
phri_brier                                                   0.076712
phri_logloss                                                 0.236005
phri_ece10                                                   0.029796
pit_ks_stat                                                  0.019724
pit_ks_pvalue                                                0.841841
selected_configs    [H64_K2_D10, H32_K3_D10, H64_K2_D10, H32_K1_D1...
dtype: object

### Interpretation boundary
These numbers are **synthetic out-of-farm development metrics**, not field performance. The validation protocol is real; the underlying pond and biological labels remain synthetic.

The PIT KS result is consistent with no strong deviation from uniformity in this synthetic OOF run, while the Brier/log-loss/ECE quantify the binary PHRI probability derived from the predictive distribution.

## 5. Selected final configuration and full-data refit

`H64_K2_D10` won validation in four of six outer folds and had the best mean validation NLL across the candidate set. The final deployment artifact therefore uses **64 LSTM hidden units, 2 Gaussian components, dropout 0.10**.

The final model is refit on all 960 synthetic cycles for 74 epochs, the median best-epoch scale from the winning outer-fold selections. Test metrics are **not** computed on this full-data refit; all reported performance comes from the out-of-fold predictions above.

In [7]:
model_card = json.loads((TRAINED / "forecast_model_trained.json").read_text())
model_card


{'execution_mode': 'pytorch_trained_mdn_lstm',
 'artifact': 'mdn_lstm_core7_trained.pt',
 'input_variables': ['temperature_c',
  'dissolved_oxygen_mg_l',
  'ph',
  'salinity_ppt',
  'ammonia_mg_l',
  'nitrite_mg_l',
  'turbidity_ntu'],
 'input_variable_count': 7,
 'lookback_days': 60,
 'forecast_target': 'end-cycle harvest_health distribution',
 'phri_definition': 'P(harvest_health < H_star | observations through Day 60)',
 'H_star': 0.7864821387045725,
 'architecture': {'lstm_hidden_size': 64,
  'lstm_layers': 1,
  'mixture_components': 2,
  'dropout': 0.1,
  'mu_link': 'sigmoid',
  'sigma_link': 'softplus + 0.005',
  'state_normalization': 'LayerNorm'},
 'training': {'optimizer': 'AdamW',
  'learning_rate': 0.001,
  'weight_decay': 0.0001,
  'gradient_clip_norm': 1.0,
  'final_fit_epochs': 74,
  'epoch_choice': 'median of best epochs from outer folds where the most frequently selected configuration H64_K2_D10 won validation',
  'final_train_nll': -2.0352602005004883},
 'validation': 

## 6. Integrity note on the previous downstream notebooks

The previous `04_monte_carlo_pricing.ipynb` and `05_evaluation_basis_risk.ipynb` reconstructed pre-existing benchmark targets; they did **not** constitute independent evaluation of this trained model. They have therefore been moved to `legacy_benchmark_reconstruction/` in this corrected package and must not be cited as trained-model performance.

A new Monte Carlo/pricing stage should consume predictions from `mdn_lstm_core7_trained.pt` / out-of-fold evidence rather than calibrating synthetic probabilities to predetermined AUC/Brier/log-loss targets.

## References

Bishop, C. M. (1994). *Mixture Density Networks* (NCRG/94/004). Aston University.

Gneiting, T., & Raftery, A. E. (2007). Strictly proper scoring rules, prediction, and estimation. *Journal of the American Statistical Association, 102*(477), 359–378. https://doi.org/10.1198/016214506000001437

Hochreiter, S., & Schmidhuber, J. (1997). Long short-term memory. *Neural Computation, 9*(8), 1735–1780. https://doi.org/10.1162/neco.1997.9.8.1735

Loshchilov, I., & Hutter, F. (2019). Decoupled weight decay regularization. *International Conference on Learning Representations (ICLR).* 

Pascanu, R., Mikolov, T., & Bengio, Y. (2013). On the difficulty of training recurrent neural networks. *Proceedings of the 30th International Conference on Machine Learning*, 1310–1318.

Roberts, D. R., et al. (2017). Cross-validation strategies for data with temporal, spatial, hierarchical, or phylogenetic structure. *Ecography, 40*(8), 913–929. https://doi.org/10.1111/ecog.02881
